In [7]:
library(dplyr)
library(ggplot2)
library(brms)

In [8]:
# Read the dataset
data <- read.csv("dataset.csv")

# Check the column names first
names(data)
# Select the required observations
data_analysis <- data %>%
  filter(
    !is.na(`body mass`),
    !is.na(`metabolic rate`),
    `metabolic rate - method` == "basal metabolic rate",
    `body mass` > 0,
    `metabolic rate` > 0
  ) %>%
  filter(class %in% c("Aves", "Mammalia")) %>%
  mutate(
    log_mass = log(`body mass`),
    log_metabolic_rate = log(`metabolic rate`)
  )
unique(data$class)
ggplot(data_analysis,
       aes(x = `body mass`,
           y = `metabolic rate`,
           color = class)) +
  
  geom_point(alpha = 0.6, size = 2) +
  
  scale_x_log10() +
  scale_y_log10() +
  
  labs(
    title = "Basal Metabolic Rate vs Body Mass",
    x = "Body Mass (kg)",
    y = "Metabolic Rate (W)",
    color = "Taxonomic Group"
  ) +
  
  theme_minimal()
# Install once if needed:
# install.packages("brms")


model_data <- data_analysis %>%
  select(log_mass, log_metabolic_rate) %>%
  na.omit()
nrow(model_data)
model_beta <- brm(
  formula = log_metabolic_rate ~ log_mass,
  
  data = model_data,
  
  family = gaussian(),
  
  prior = c(
    prior(normal(0, 10), class = "Intercept"),
    prior(normal(0, 2), class = "b"),
    prior(student_t(3, 0, 2.5), class = "sigma")
  ),
  
  chains = 4,
  iter = 4000,
  warmup = 2000,
  cores = 4,
  seed = 1234
)
posterior_beta <- as_draws_df(model_beta)

beta_samples <- posterior_beta$b_log_mass
summary(beta_samples)
quantile(
  beta_samples,
  probs = c(0.025, 0.5, 0.975)
)
ggplot(
  data.frame(beta = beta_samples),
  aes(x = beta)
) +
  geom_density() +
  geom_vline(
    xintercept = 0.75,
    linetype = "dashed"
  ) +
  labs(
    title = "Posterior Distribution of the Allometric Exponent",
    x = expression(beta),
    y = "Posterior Density"
  ) +
  theme_minimal()

[1] "phylum"                              
 [2] "class"                               
 [3] "order"                               
 [4] "family"                              
 [5] "genus"                               
 [6] "species"                             
 [7] "specificEpithet"                     
 [8] "sex"                                 
 [9] "sampleSizeValue"                     
[10] "inTextReference"                     
[11] "publicationYear"                     
[12] "fullReference"                       
[13] "body.mass"                           
[14] "body.mass...units"                   
[15] "body.mass...minimum"                 
[16] "body.mass...maximum"                 
[17] "body.mass...method"                  
[18] "body.mass...comments"                
[19] "body.mass...metadata.comment"        
[20] "original.body.mass"                  
[21] "original.body.mass...units"          
[22] "metabolic.rate"                      
[23] "metabolic.rate...units"              
[24] "metabolic.rate...minimum"            
[25] "metabolic.rate...maximum"            
[26] "metabolic.rate...method"             
[27] "metabolic.rate...comments"           
[28] "metabolic.rate...metadata.comment"   
[29] "original.metabolic.rate"             
[30] "original.metabolic.rate...units"     
[31] "original.respiratoryQuotient"        
[32] "original.temperature"                
[33] "mass.specific.metabolic.rate"        
[34] "mass.specific.metabolic.rate...units"
[35] "brain.size"                          
[36] "brain.size...units"                  
[37] "brain.size...minimum"                
[38] "brain.size...maximum"                
[39] "brain.size...method"                 
[40] "brain.size...comments"               
[41] "brain.size...metadata.comment"       
[42] "original.brain.size"                 
[43] "original.brain.size...units"

ERROR: [1m[33mError[39m in `filter()`:[22m
[1m[22m[36mℹ[39m In argument: `!is.na(`body mass`)`.
[1mCaused by error:[22m
[33m![39m object 'body mass' not found
